# Parte I · La base teórica y conceptual

Todo lo que sigue gira alrededor de **una** pregunta: *¿cuánta variación genética hay en estas poblaciones y qué podemos afirmar con ella?* La dificultad no está en las fórmulas —son aritmética simple— sino en tres cosas: **qué pregunta responde cada número**, **cuánta incertidumbre tiene** y **qué sesgos lo empujan**.

## 1 · Qué se mide: frecuencias

Un **SNP** es una posición del genoma donde los individuos difieren en un nucleótido. En cada individuo hay dos copias (es diploide), así que un genotipo puede ser homocigoto de referencia (0), heterocigoto (1) u homocigoto alternativo (2). Guardamos eso como **dosis**: el número de copias del alelo alternativo. Esa decisión no es cosmética: con números se hace aritmética, y de ahí salen todas las cantidades de la sección.

De una muestra de $n$ individuos salen dos tipos de frecuencia:

- **Frecuencia genotípica**: proporción de individuos con cada genotipo. Es un **conteo** (6 heterocigotos de 12 → 0.5). No requiere modelo ni supuestos.
- **Frecuencia alélica**: proporción de *copias* de cada alelo. Hay $2n$ copias, y $p$, la frecuencia del alelo alternativo, es simplemente la **dosis media dividida por 2**. Ejemplo: dosis `0, 1, 1, 2, 0` → 4 copias alternativas de 10 → $p = 0.4$.

**La consecuencia práctica:** la columna de un SNP, promediada, *es* su frecuencia alélica. Por eso un error de orientación de la matriz (filas por columnas) no produce ningún error de código, solo números equivocados sin aviso: la Sección 1 insistía en verificar la alineación con un `stopifnot` justamente por esto.

## 2 · El modelo nulo: Hardy-Weinberg

Para saber si un número es alto, bajo o raro hace falta un punto de comparación. En genética de poblaciones ese punto es **Hardy-Weinberg (HW)**: si el apareamiento es al azar, la población es grande y no hay migración, mutación ni selección, entonces las frecuencias alélicas no cambian entre generaciones y los genotipos se reparten según

$$f(AA)=p^2 \qquad f(Aa)=2pq \qquad f(aa)=q^2 \qquad p^2+2pq+q^2=1$$

Con $p = 0.333$: $p^2 = 0.111$, $2pq = 0.444$, $q^2 = 0.444$ (en 12 individuos, 1.3 / 5.3 / 5.3 esperados).

HW **no es una ley de la naturaleza**: es el *modelo nulo*, el escenario "no pasa nada especial". Cuando los datos se apartan, el interés está en explicar **por qué** —endogamia, subestructura, selección, errores de genotipado—. Y como es un modelo, siempre se aparta un poco: la pregunta nunca es *¿se cumple?*, sino *¿se aparta más de lo que el azar explica?*.

**Dos consecuencias que se olvidan:** el máximo de $H_e = 2pq$ es **0.5**, y solo con $p = q = 0.5$. Por eso una $H_e$ de 0.33 no es "dos tercios de diversa": los marcadores simplemente están lejos del 50-50. Y como todo depende de $p$, **cualquier filtro que cambie $p$ cambia la diversidad** (esto reaparece en el punto 6).

## 3 · Los tres números: Ho, He y Fis

- **Ho (heterocigosidad observada)**: proporción de individuos heterocigotos, promedio sobre marcadores. Se **cuenta**: 6 de 12 → $H_o = 0.5$.
- **He (heterocigosidad esperada)**: lo que HW predice con las frecuencias de esa población: $H_e = 2pq$. Con $p = 0.333$ → $0.444$.
- **Fis (índice de endogamia)**: cuánto se desvía lo observado de lo esperado, en proporción: $F_{is} = 1 - H_o/H_e$. Con esos números: $1 - 0.5/0.444 = -0.126$.

El **signo** es la información:

| $F_{is}$ | Lectura | Causas típicas |
|---|---|---|
| $> 0$ | Déficit de heterocigotos | endogamia, subestructura (efecto Wahlund), alelos nulos o *dropout*, selección |
| $\approx 0$ | Coincide con HW | apareamiento aproximadamente al azar |
| $< 0$ | Exceso de heterocigotos | entrecruzamiento, apareamiento entre no parientes, $n$ pequeño |

Y la distinción que ordena toda la sección: **He describe la variación; Fis describe el apareamiento.** Son dos preguntas distintas y ninguna reemplaza a la otra: una población puede tener mucha variación y estar endogámica al mismo tiempo.

> **Cuidado con la interpretación automática.** Un $F_{is} > 0$ no demuestra endogamia. Los **alelos nulos** (un alelo que no amplifica y se lee como homocigoto) y el **dropout** producen exactamente el mismo signo. El número abre una pregunta; no la cierra. Y en marcadores dominantes (AFLP) ni siquiera se puede calcular $F_{is}$ individual, porque no se distingue heterocigoto de homocigoto: es un límite del marcador, no del análisis.



## 4 · Estimar con muestras chicas: la corrección de Nei

Si calculas $H_e$ con la **misma** muestra con la que estimaste $p$, el resultado subestima la heterocigosidad de la población: los alelos raros quedan subrepresentados por azar del muestreo y $2\hat{p}(1-\hat{p})$ es un estimador sesgado hacia abajo. La corrección depende solo del tamaño:

$$H_{e,\text{insesgada}} = 2\hat{p}(1-\hat{p})\times\frac{2n}{2n-1}$$

| $n$ individuos | Copias $2n$ | Factor | Si $H_e$ simple = 0.300 |
|---|---|---|---|
| 4 | 8 | 1.143 | 0.343 |
| 6 | 12 | 1.091 | 0.327 |
| 11 | 22 | 1.048 | 0.314 |
| 12 | 24 | 1.043 | 0.313 |

Con 6 individuos el factor es 1.09: casi un 10 % de diferencia que **no es biología**. De ahí una regla que vale para todo el curso: dos poblaciones —o dos estudios— con distinto $n$ no se comparan en crudo.

## 5 · ¿Cuánto confiar? Incertidumbre y bootstrap

Un número solo no es un resultado: toda estimación tiene un rango de valores plausibles. El **bootstrap** lo aproxima remuestreando los datos que ya tienes:

1. Tomar los marcadores y **remuestrearlos con reemplazo** (algunos entran dos veces, otros ninguna).
2. Recalcular la estadística con esa muestra nueva.
3. Repetir $B$ veces ($B = 200$ acá; 1 000 o más en un trabajo real, con semilla fija para que sea reproducible).
4. Tomar el rango entre los percentiles 2.5 y 97.5: un **intervalo de confianza aproximado del 95 %**.

**Qué mide y qué no.** Mide cuánto depende la estimación de **qué marcadores** te tocaron. **No** mide la incertidumbre por haber muestreado 12 árboles en vez de 50: ese es otro experimento. Confundir ambos es el error más común al reportar intervalos.

**Cómo se lee.** Si los intervalos de dos poblaciones **se solapan**, con esos datos la diferencia no está clara. Si no se solapan, hay una diferencia consistente con estos marcadores — y se reporta con **magnitud e intervalo**, no con la palabra "significativo".



## 6 · Los sesgos que hay que saber nombrar

**a) Tamaño muestral.** Con pocos individuos, $H_e$ se subestima y la dispersión entre poblaciones crece: cada estimación es más ruidosa. Graficar la estimación contra $n$ es la forma de **elegir el $n$ mínimo aceptable** en un estudio propio, en vez de heredarlo de la literatura.

**b) MAF y sesgo de verificación.** El **MAF** es la frecuencia del alelo menos común, $\min(p,q)$, entre 0 y 0.5. Como $H_e = 2pq$ crece con $p$ hasta 0.5, subir el umbral de MAF elimina los marcadores de menor $H_e$ y **la diversidad promedio sube**: no cambió la población, cambió el filtro. En nuestros datos: 1 174 SNP con MAF > 0.05 → $H_e$ 0.325; 1 017 con MAF > 0.10 → 0.356; 725 con MAF > 0.20 → 0.405.

Detrás está el **sesgo de verificación** (*ascertainment bias*): un panel de SNP no es una muestra al azar del genoma. Los marcadores se descubren y eligen con algún criterio (secuenciar pocos individuos, diseñar un arreglo, exigir calidad), y si ese criterio favorece variantes comunes, la diversidad queda subestimada por construcción. Por eso "comparar con la literatura" exige mismo **tipo de marcador** y mismo **filtro**.

**c) Riqueza alélica y alelos privados.** La **riqueza** es el número de alelos distintos; un **alelo privado** está en una población o región y ausente en las demás (señal de aislamiento o de historia independiente). Ambos **crecen con $n$**, porque más individuos = más variantes raras detectadas. Comparar 58 individuos contra 94 sin corregir es comparar esfuerzo de muestreo. La solución estándar es la **rarefacción**: recalcular al $n$ más pequeño del conjunto.

## 7 · Geografía: las poblaciones no son observaciones independientes

Un modelo como `lm(He ~ lat)` supone residuos independientes. Veinte poblaciones repartidas en 2 000 km no lo son: las vecinas comparten historia, flujo génico y refugios. Eso es **pseudorreplicación** —el mismo problema que contar células como si fueran donantes—, y su consecuencia es concreta: **los $p$-valores ingenuos son demasiado optimistas**. Un $p = 10^{-5}$ sobre poblaciones correlacionadas no es la misma evidencia que sobre sitios independientes.

Qué hacer, en orden de esfuerzo: **escribirlo** en Limitaciones y hablar de evidencia sugerente; **permutar** de forma restringida (dentro de regiones); o **modelar** la dependencia (`nlme::gls` con `corExp`, o un enfoque de aislamiento por distancia, que es material de la Sección 3).

## 8 · Alcance: lo que esta sección sí y no responde

| Pregunta | Se responde aquí | Dónde |
|---|---|---|
| ¿Cuánta variación hay dentro de cada población? | Sí | $H_e$, $H_o$, riqueza, proporción de polimórficos |
| ¿Cómo se aparean esos individuos? | Parcialmente | $F_{is}$ (con las salvedades del punto 3) |
| ¿Cuánta confianza tiene cada número? | Sí, en parte | bootstrap sobre marcadores |
| ¿Están diferenciadas las poblaciones entre sí? | **No** | Sección 3: $F_{st}$, AMOVA, PCA/DAPC |
| ¿Los marcadores son independientes entre sí? | **No** | Sección 3: LD y poda |
| ¿Hay asociación genotipo–ambiente? | **No** | Sección 6: RDA |

Saber qué **no** responde un análisis es parte del análisis: es lo que va en Limitaciones.

### Lo esencial, en cinco frases

1. $H_e$ mide **variación**; $F_{is}$ mide **apareamiento**. Dos preguntas, dos números.
2. Hardy-Weinberg es el **modelo nulo**: apartarse de él es el dato, no el fallo.
3. El máximo de $H_e$ es **0.5**, y depende de $p$: los filtros que cambian $p$ cambian la diversidad.
4. **Un número sin incertidumbre no es un resultado**; y el bootstrap cubre solo una parte de la incertidumbre.
5. La independencia espacial casi siempre se viola: con poblaciones correlacionadas, un $p$-valor ingenuo es optimista.


## Hoja de fórmulas

| Cantidad | Fórmula | Responde | Nota para usarla |
|---|---|---|---|
| Frecuencia alélica | $p = \bar{d}/2$ | ¿Qué proporción de copias es alternativa? | Sale de la dosis media de la columna |
| Hardy-Weinberg | $p^2,\ 2pq,\ q^2$ | ¿Qué esperarías si nada especial pasa? | Requiere apareamiento al azar, población grande, sin migración/mutación/selección |
| $H_o$ | heterocigotos / $n$ | ¿Cuántos heterocigotos hay? | Conteo, sin modelo |
| $H_e$ | $2pq$ | ¿Cuánta variación esperar? | Máximo 0.5 |
| $H_e$ insesgada | $2pq\cdot\frac{2n}{2n-1}$ | Corregir $n$ chico | Compara solo con la misma corrección |
| $F_{is}$ | $1-H_o/H_e$ | ¿Cómo se aparean? | Signo, no diagnóstico: alelos nulos dan lo mismo |
| MAF | $\min(p,q)$ | ¿Qué tan variable es el marcador? | Sube el umbral → sube $H_e$ |
| IC 95 % (bootstrap) | percentiles 2.5–97.5 | ¿Cuánto depende de los marcadores? | No cubre el muestreo de individuos |
| Riqueza alélica | nº de alelos distintos | ¿Cuántos alelos hay? | Rarefaccionar antes de comparar |

## Mapa: qué concepto sostiene cada análisis

Los siete análisis de la Parte II **no introducen nada nuevo**. Antes de correrlos, conviene tener claro qué se está aplicando y qué se espera ver.

| Análisis | Concepto que aplica | Qué produce |
|---|---|---|
| §2 Diversidad por población | 1, 3, 4 | $H_o$, $H_e$, $F_{is}$ y proporción de polimórficos por población |
| §3 Gradiente con la latitud | 1, 3, 7 | Una recta, $R^2$ y una correlación, con sus advertencias |
| §4 Bootstrap | 5 | Intervalos de confianza para dos poblaciones extremas |
| §5 Regiones | 3, 6 | Medias por región y un test, más el problema del $n$ desigual |
| §6 Diversidad individual | 1, 3 | Detección de individuos atípicos |
| §7 Alelos privados | 6c | Conteo por región, con la advertencia de rarefacción |


# Parte II · Análisis

**Preparación.** Carga los datos limpios de la Sección 1 y calcula los promedios de latitud, longitud y altitud por población.


In [ ]:
# ---- Preparación (ejecutar primero) ----
# Tamaño de gráficos en Jupyter
options(repr.plot.width = 7, repr.plot.height = 4.5, repr.plot.res = 110)

source("R/helpers.R")
d <- load_clean(); G <- d$G; meta <- d$meta
meta$region <- assign_region(meta$lat)
pops <- aggregate(cbind(lat, lon, alt) ~ pop, data = meta, FUN = mean)

## §2 · Diversidad por población

**Antes de mirar la tabla:** ¿en qué rango esperas $H_e$ (recuerda: máximo 0.5)? ¿Y $F_{is}$ positivo o negativo? Anótalo.


In [ ]:
div <- diversity_by_pop(G, meta$pop)
div <- merge(div, pops, by = "pop")
div <- div[order(div$lat, decreasing = TRUE), ]   # de norte a sur
round_df <- function(x, k = 3) { x[sapply(x, is.numeric)] <- round(x[sapply(x, is.numeric)], k); x }
head(round_df(div), 8)

## §3 · ¿Hay un gradiente con la latitud?

**Antes de correr:** la latitud va en negativos, así que "más al norte" es "menos negativo". Si la diversidad baja hacia el sur, ¿qué signo tendrá la pendiente?


In [ ]:
ggplot(div, aes(lat, He)) + geom_point(size = 3) + geom_smooth(method = "lm", se = TRUE) +
  labs(x = "Latitud (°)", y = "He") 

El ajuste lineal y la correlación de Spearman (que no supone linealidad):


In [ ]:
m <- lm(He ~ lat, data = div)
summary(m)$coefficients
cor.test(div$lat, div$He, method = "spearman")

## §4 · Incertidumbre: bootstrap sobre SNP

**Antes de correr:** ¿esperas que los intervalos de L01 y L20 se solapen? ¿Qué significaría cada caso?


In [ ]:
set.seed(1)
boot_he <- function(Gk, B = 200) {
  replicate(B, {
    cols <- sample(ncol(Gk), replace = TRUE)
    diversity_by_pop(Gk[, cols], rep("x", nrow(Gk)))$He
  })
}
# ejemplo para una población (L01) y otra (L20)
b1  <- boot_he(G[meta$pop == "L01", ]); b20 <- boot_he(G[meta$pop == "L20", ])
rbind(L01 = quantile(b1, c(.025, .5, .975)), L20 = quantile(b20, c(.025, .5, .975)))

## §5 · Comparación entre regiones

**Antes de correr:** ¿qué región esperas con más dispersión? Y mirando los tamaños de muestra, ¿la comparación es justa?


In [ ]:
div$region <- assign_region(div$lat)
aggregate(cbind(Ho, He, Fis) ~ region, data = div, FUN = mean)
kruskal.test(He ~ region, data = div)
ggplot(div, aes(region, He, fill = region)) + geom_boxplot(alpha = .6, show.legend = FALSE) +
  geom_jitter(width = .1)

## §6 · Diversidad individual

**Antes de correr:** ¿habrá individuos fuera de rango? ¿Qué podría explicarlos, y qué NO podrías concluir solo con este gráfico?


In [ ]:
het_ind <- rowMeans(G == 1, na.rm = TRUE)       # heterocigosidad multilocus
ind_df  <- data.frame(meta, het = het_ind)
ggplot(ind_df, aes(reorder(pop, -lat), het)) + geom_boxplot() +
  labs(x = "Población (norte → sur)", y = "Heterocigosidad individual") +
  theme(axis.text.x = element_text(angle = 90, vjust = .5))

## §7 · Alelos privados por región

**Antes de correr:** con una estructura en gradiente continuo, ¿cuántos alelos exclusivos esperas? ¿Y qué pasa con el $n$ de cada región?


In [ ]:
reg <- factor(meta$region)
Pr  <- t(sapply(levels(reg), function(r) colMeans(G[reg == r, ], na.rm = TRUE) / 2))
privados <- sapply(seq_len(nrow(Pr)), function(i) {
  otros <- Pr[-i, , drop = FALSE]
  sum(Pr[i, ] > 0 & colSums(otros > 0) == 0) +      # ALT presente solo aquí
  sum(Pr[i, ] < 1 & colSums(otros < 1) == 0)        # REF presente solo aquí
})
setNames(privados, levels(reg))
table(reg)     # ¡el esfuerzo de muestreo difiere!

---

# Ejercicios

Los cuatro ejercicios apuntan a lo mismo: que la conclusión no dependa del primer número que apareció. Cada uno trae qué hacer, una pista y cómo saber que está bien; la "idea clave" es la que va al informe.

**E1 · Tres métricas, tres conclusiones posibles.** Ajusta `lm(He ~ lat)`, `lm(Ho ~ lat)` y `lm(Fis ~ lat)`; grafica los tres y compara pendientes y $R^2$.

- **Pista:** parte de `div`; `lm()` + `coef(summary(...))` da coeficientes y $p$ en una tabla.
- **Cómo saber que está bien:** puedes nombrar una métrica con gradiente y otra sin él. Si las tres coinciden, sospecha del filtro antes que de la biología.
- **Idea clave:** se reportan las tres; elegir la que "salió mejor" es sesgo de confirmación.


In [ ]:
# TU CÓDIGO AQUÍ

**E2 · Sesgo por tamaño de muestra.** Submuestrea 4, 6, 8 y 12 individuos por población (20 réplicas cada uno), recalcula $H_e$ y $F_{is}$, y grafica media y dispersión versus $n$.

- **Pista:** `split(seq_len(nrow(G)), meta[["pop"]])` da los índices por población; `sample()` elige los individuos.
- **Cómo saber que está bien:** la dispersión se angosta al crecer $n$ y la media de $H_e$ sube levemente (menos sesgo y corrección de Nei).
- **Idea clave:** el $n$ mínimo aceptable se define **mirando esa curva**, no copiándolo de un paper.


In [ ]:
# Pista: split(seq_len(nrow(G)), meta$pop) + sample()

**E3 · Sesgo de verificación (MAF).** Recalcula la diversidad con MAF > 0.05, 0.10 y 0.20 y compara la relación $H_e \sim$ latitud.

- **Pista:** la frecuencia global es `colMeans(G, na.rm = TRUE)/2`; el MAF es `pmin(p, 1-p)`; filtra columnas con `G[, keep]`.
- **Cómo saber que está bien:** quedan menos marcadores, la $H_e$ media sube y la **dirección** del gradiente se mantiene (los valores están en el punto 6b para comparar).
- **Idea clave:** el filtro mueve el nivel, no la biología; pero un lector solo puede comparar tu $H_e$ con otro estudio si el filtro coincide.


In [ ]:
# TU CÓDIGO AQUÍ

**E4 · Dependencia espacial.** Reemplaza la regresión por una versión que considere la autocorrelación espacial (`nlme::gls` con `corExp` sobre lat/lon, o una permutación que permute poblaciones **dentro** de regiones). Discute qué cambia.

- **Pista:** guarda los coeficientes y el $p$ del modelo ingenuo antes de cambiarlo; la comparación **es** el resultado.
- **Cómo saber que está bien:** el $p$ se vuelve menos optimista (o el intervalo se ensancha) y puedes explicar por qué.
- **Idea clave:** con dependencia espacial, el $p$-valor ingenuo sobreestima la evidencia.


In [ ]:
# TU CÓDIGO AQUÍ

## Errores y dudas frecuentes

- **"Mi bootstrap da intervalos muy anchos."** Esperable con pocos individuos y pocos marcadores; es información sobre el diseño, no un error.
- **"$F_{is}$ me da negativo en una población."** Posible con $n$ pequeño; no interpretes un valor aislado.
- **"¿Por qué la $H_e$ insesgada?"** Porque estimar $p$ con la misma muestra subestima $2pq$ (punto 4).
- **"$F_{is}$ positivo ⇒ endogamia."** No: también lo producen subestructura, alelos nulos y *dropout* (punto 3).
- **"$p < 0.05$ ⇒ hay gradiente."** Con poblaciones correlacionadas, no (punto 7).
- **"$H_e$ bajó al filtrar más."** Si subiste el MAF, debería **subir**; si bajó, revisa si cambiaste otra cosa (call rate, conjunto de marcadores).
- **"¿Puedo comparar con la literatura?"** Solo con el mismo tipo de marcador, el mismo filtro y el mismo tipo de corrección.

## Antes de entregar la Tarea

- [ ] ¿El gráfico tiene **intervalos** (bootstrap), no solo puntos?
- [ ] ¿La conclusión tiene **magnitud + dirección + incertidumbre**, sin la palabra "significativo"?
- [ ] ¿Las Limitaciones nombran **dependencia espacial**, **$n$ desigual entre regiones** y **efecto del filtro de MAF**?


## Tarea (se revisa al comienzo de la Sección 3)

1. **Gráfico publicable** de $H_e$ por población, ordenado por latitud, con **barras de error bootstrap**. Ejes rotulados, y una nota al pie con $B$ y la semilla usada.
2. **Una frase de conclusión sin "significativo"**: magnitud, dirección e incertidumbre. Ejemplo: *"observamos una tendencia a menor diversidad hacia el sur (pendiente $x$, IC 95 % $[a;b]$), con incertidumbre y dependencia espacial entre poblaciones"*.
3. **Tres líneas de Limitaciones** que nombren, como mínimo: dependencia espacial, desigualdad de $n$ entre regiones y efecto del filtro de MAF sobre $H_e$.

*Para discutir al inicio de la Sección 3:* si observaras menor diversidad al sur, ¿qué hipótesis alternativas lo explicarían —efectos fundadores post-glaciares, menor tamaño poblacional, refugios, error de muestreo, sesgo de verificación— y **qué dato adicional** distinguiría una de otra?
